In [1]:
import pandas as pd
import networkx as nx
import os
import numpy as np
from collections import deque
from gene_to_uniprot import convert_gene_list#genes--uniprot

DATA_PATH = r"C:\Users\Nisrin Fariss Lamine\Downloads\tfm"

def cargar_datos():
    print("Cargando BioGRID...")
    df_edges = pd.read_csv(os.path.join(DATA_PATH, "biogrid_edges.csv"))
    G = nx.from_pandas_edgelist(df_edges, source="source", target="target")

    print("Cargando DrugBank limpio...")
    df_drug = pd.read_csv(os.path.join(DATA_PATH, "drugbank_targets_clean.csv"))

    print("Cargando Enfermedades...")
    df_disorders = pd.read_csv(os.path.join(DATA_PATH, "disorder_genes.csv"), sep=";")

    return G, df_drug, df_disorders


In [2]:
def bfs_multifuente(grafo, origenes):
    
    distancias = {nodo: float("inf") for nodo in grafo.nodes()}#dist inf,nodos no visitados
    cola = deque()

    for o in origenes:
        if o in distancias:#si esta en nodos no visitados
            distancias[o] = 0# a simismo 
            cola.append(o)

    while cola:
        actual = cola.popleft()
        for vecino in grafo.neighbors(actual):
            if distancias[vecino] == float("inf"):
                distancias[vecino] = distancias[actual] + 1
                cola.append(vecino)

    return distancias#diccionario de nodo y su distancia minima desde  laenfermedad al resto de nodos


In [3]:
G = nx.Graph()
G.add_edges_from([
    ("A", "B"),
    ("B", "C"),
    ("C", "D"),
    ("A", "E")
])
origenes = ["A"]

dist = bfs_multifuente(G, origenes)

print(dist)

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 1}


In [4]:
def construir_bins_por_grado(grafo, tamaño_bin=50):
    
    grados = {nodo: grafo.degree(nodo) for nodo in grafo.nodes()}#conexion de cada nodo
    bins = {}
    for proteina, grado in grados.items():
        bin_id = grado // tamaño_bin #ver a q bin corresponde, eneteros
        bins.setdefault(bin_id, []).append(proteina)
        
    return grados, bins

In [5]:
grados, bins = construir_bins_por_grado(G, tamaño_bin=2)

print("Grados:", grados)
print("Bins:", bins)

Grados: {'A': 2, 'B': 2, 'C': 2, 'D': 1, 'E': 1}
Bins: {1: ['A', 'B', 'C'], 0: ['D', 'E']}


In [6]:
def seleccionar_enfermedad_valida(df_disorders, G):#conversion genes--uniprot
    for enf in df_disorders["disorder"].unique():

        genes_str = df_disorders[df_disorders["disorder"] == enf]["gene_symb"].values[0]#todos los genes
        genes_simbolos = [g.strip() for g in genes_str.split(",") if g.strip()]#separa lista genes
        conversion = convert_gene_list(genes_simbolos)
        genes_uniprot = [u for u in conversion.values() if u is not None]
        genes_presentes = [u for u in genes_uniprot if u in G.nodes()]#solo las presentes en la red

        if len(genes_presentes) >= 1:
            return enf, genes_presentes

    return None, []

In [7]:
G, df_drug, df_disorders = cargar_datos()

enf, genes = seleccionar_enfermedad_valida(df_disorders, G)
print("-------------------------------")

print("Enfermedad seleccionada:", enf)
print("Genes válidos:", genes)
print("Número de genes:", len(genes))



Cargando BioGRID...
Cargando DrugBank limpio...
Cargando Enfermedades...
-------------------------------
Enfermedad seleccionada: 3-M syndrome
Genes válidos: ['Q14999', 'O75147', 'Q9H0W5', 'Q9H0W5']
Número de genes: 4


In [9]:
grados, bins = construir_bins_por_grado(G)

In [8]:
def generar_dianas_aleatorias(dianas_reales, grados, bins, tamaño_bin=50):
    
    aleatorias = []

    for diana in dianas_reales:

        grado = grados.get(diana)#grado de esa diana

        if grado is None:# si no esta en el grafo se ignora
            continue

        bin_id = grado // tamaño_bin

        if bin_id not in bins:#bin inexistente se ignora
            continue
        aleatorias.append(random.choice(bins[bin_id]))#selecciona una proteína aleatoria del mismo bin


    return aleatorias

In [11]:

import random
# coger un fármaco real
drug = df_drug["DrugBank_ID"].iloc[0]

dianas_reales = list(
    set(df_drug[df_drug["DrugBank_ID"] == drug]["UniProt_ID"])
)

print("Dianas reales:", dianas_reales[:5])

# ejecutar función
aleatorias = generar_dianas_aleatorias(dianas_reales, grados, bins)

print("Dianas aleatorias:", aleatorias[:5])

Dianas reales: ['P00734']
Dianas aleatorias: ['Q6UWV7']


In [12]:
print(len(dianas_reales), len(aleatorias))

1 1


In [13]:
for real, rand in zip(dianas_reales[:5], aleatorias[:5]):
    print(f"{real} ({grados.get(real)}) → {rand} ({grados.get(rand)})")#mismo bin

P00734 (41) → Q6UWV7 (3)


In [15]:
def distancia_media_conjunto(dianas, distancias):
    
    #la distancia de cada diana desde el diccionario de distancias
    valores = [distancias.get(d, float("inf")) for d in dianas]

    # no alcanzables(infinito)
    valores = [v for v in valores if v != float("inf")]

    return sum(valores) / len(valores) if valores else None




In [17]:

# distancias desde enfermedad
dist_ref = bfs_multifuente(G, genes)

# coger un fármaco
drug = df_drug["DrugBank_ID"].iloc[0]

dianas = list(df_drug[df_drug["DrugBank_ID"] == drug]["UniProt_ID"])

# calcular media
media = distancia_media_conjunto(dianas, dist_ref)

print("Distancia media:", media)

Distancia media: 2.0


In [20]:
print(set(dist_ref.keys()) == set(G.nodes()))# todos los nodos del grafo

True


In [24]:
def proximidad_estadistica(dianas, distancias_ref, grados, bins, repeticiones=200):
    
    dist_obs = distancia_media_conjunto(dianas, distancias_ref)#dist real

    if dist_obs is None:
        return None, None, None, None, None

    dist_aleatorias = []

    for _ in range(repeticiones): #simulaciones =

        d_rand = generar_dianas_aleatorias(dianas, grados, bins)

        dist_rand = distancia_media_conjunto(d_rand, distancias_ref)
        if dist_rand is not None:
            dist_aleatorias.append(dist_rand)

    if len(dist_aleatorias) < 3: #en caso de pocas observaciones
        return dist_obs, None, None, None, None

    media = np.mean(dist_aleatorias)
    desviacion = np.std(dist_aleatorias) if np.std(dist_aleatorias) > 0 else 1e-9#q no divida entre 0

    z = (dist_obs - media) / desviacion#  cuántas desviaciones está la real respecto a la media
    p = (1 + sum(x <= dist_obs for x in dist_aleatorias)) / (1 + len(dist_aleatorias))

    return dist_obs, media, desviacion, z, p

In [21]:
print("Enfermedad:", enf)
print("Genes:", genes)

Enfermedad: 3-M syndrome
Genes: ['Q14999', 'O75147', 'Q9H0W5', 'Q9H0W5']


In [22]:
for drug in df_drug["DrugBank_ID"].unique():
    dianas = list(df_drug[df_drug["DrugBank_ID"] == drug]["UniProt_ID"])
    dianas_en_red = [d for d in dianas if d in G.nodes()]
    if len(dianas_en_red) >= 1:
        break

print("\nFármaco:", drug)
print("Dianas en red:", dianas_en_red)


Fármaco: DB00001
Dianas en red: ['P00734']


In [26]:

resultado = proximidad_estadistica(dianas_en_red,dist_ref,grados,bins,repeticiones=200 )

print("dist_obs:", resultado[0])
print("media:", resultado[1])
print("std:", resultado[2])
print("z:", resultado[3])
print("p:", resultado[4])


RESULTADOS:
dist_obs: 2.0
media: 2.12
std: 0.32496153618543844
z: -0.3692744729379985
p: 0.8805970149253731
